# Food Demand Forecasting â€” ML Training Pipeline
**FYP TP070073 | Sanjivan Thiyageswaran**

This notebook covers the full ML pipeline:
1. Mount Drive & load data
2. Merge datasets
3. Feature engineering
4. Time-based train/validation split
5. Train Random Forest & XGBoost
6. Evaluate (MAE, RMSE)
7. Save best model artifacts

## 1. Setup

In [ ]:
# Install xgboost if not available
!pip install xgboost --quiet

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries loaded successfully.')

## 2. Mount Google Drive & Load Dataset

Upload your dataset folder to Google Drive under `My Drive/FYP/dataset/` before running this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path if your folder structure is different
DATASET_PATH = '/content/drive/MyDrive/FYP/dataset/'

train      = pd.read_csv(DATASET_PATH + 'train.csv')
test       = pd.read_csv(DATASET_PATH + 'test.csv')
meal_info  = pd.read_csv(DATASET_PATH + 'meal_info.csv')
center_info = pd.read_csv(DATASET_PATH + 'fulfilment_center_info.csv')

print('train shape     :', train.shape)
print('test shape      :', test.shape)
print('meal_info shape :', meal_info.shape)
print('center_info shape:', center_info.shape)

## 3. Explore & Merge

In [ ]:
print('--- train.csv ---')
display(train.head())
print('Columns:', train.columns.tolist())
print('Missing values:\n', train.isnull().sum())

In [ ]:
print('--- meal_info.csv ---')
display(meal_info.head())
print('\n--- fulfilment_center_info.csv ---')
display(center_info.head())

In [ ]:
# Merge all three tables
df = train.merge(meal_info, on='meal_id', how='left')
df = df.merge(center_info, on='center_id', how='left')

print('Merged train shape:', df.shape)
display(df.head())

In [ ]:
# Brief EDA â€” target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['num_orders'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of num_orders')
axes[0].set_xlabel('num_orders')

axes[1].hist(np.log1p(df['num_orders']), bins=80, color='coral', edgecolor='white')
axes[1].set_title('Distribution of log(num_orders + 1)')
axes[1].set_xlabel('log(num_orders + 1)')

plt.tight_layout()
plt.show()

print('num_orders stats:')
print(df['num_orders'].describe())

## 4. Feature Engineering

In [ ]:
# --- 4a. Price discount feature ---
df['discount_pct'] = (df['base_price'] - df['checkout_price']) / df['base_price']
df['discount_pct'] = df['discount_pct'].clip(lower=0)  # clip negatives (price > base)

# --- 4b. Label encode categorical columns ---
label_encoders = {}
for col in ['category', 'cuisine', 'center_type']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'{col} classes: {le.classes_.tolist()}')

print('\nLabel encoding done.')

In [ ]:
# --- 4c. Lag features & rolling mean (grouped by center + meal) ---
# Sort by time first â€” critical for correct lag calculation
df = df.sort_values(['center_id', 'meal_id', 'week']).reset_index(drop=True)

grp = df.groupby(['center_id', 'meal_id'])['num_orders']

df['lag_1']        = grp.shift(1)
df['lag_2']        = grp.shift(2)
df['lag_3']        = grp.shift(3)
df['rolling_mean_3'] = grp.shift(1).rolling(window=3, min_periods=1).mean().values
df['rolling_std_3']  = grp.shift(1).rolling(window=3, min_periods=1).std().fillna(0).values

rows_before = len(df)
df = df.dropna(subset=['lag_1', 'lag_2', 'lag_3'])
print(f'Dropped {rows_before - len(df)} rows due to lag NaNs. Remaining: {len(df)}')

In [ ]:
# Define final feature set
FEATURE_COLS = [
    'week',
    'center_id',
    'meal_id',
    'checkout_price',
    'base_price',
    'discount_pct',
    'emailer_for_promotion',
    'homepage_featured',
    'op_area',
    'city_code',
    'region_code',
    'center_type_enc',
    'category_enc',
    'cuisine_enc',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_mean_3',
    'rolling_std_3',
]
TARGET_COL = 'num_orders'

print('Feature count:', len(FEATURE_COLS))
print('Features:', FEATURE_COLS)

## 5. Time-Based Train / Validation Split

We use weeks 1â€“130 for training and weeks 131â€“145 for validation.  
**Never use random split for time-series data** â€” it leaks future information.

In [ ]:
TRAIN_END_WEEK = 130

train_df = df[df['week'] <= TRAIN_END_WEEK]
val_df   = df[df['week'] >  TRAIN_END_WEEK]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]

X_val = val_df[FEATURE_COLS]
y_val = val_df[TARGET_COL]

print(f'Training set : {X_train.shape} (weeks 1â€“{TRAIN_END_WEEK})')
print(f'Validation set: {X_val.shape} (weeks {TRAIN_END_WEEK+1}â€“145)')

## 6. Train Models

In [ ]:
# --- 6a. Random Forest ---
print('Training Random Forest...')
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_val)
rf_mae   = mean_absolute_error(y_val, rf_preds)
rf_rmse  = np.sqrt(mean_squared_error(y_val, rf_preds))

print(f'Random Forest  â€” MAE: {rf_mae:.2f} | RMSE: {rf_rmse:.2f}')

In [ ]:
# --- 6b. XGBoost ---
print('Training XGBoost...')
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

xgb_preds = xgb_model.predict(X_val)
xgb_mae   = mean_absolute_error(y_val, xgb_preds)
xgb_rmse  = np.sqrt(mean_squared_error(y_val, xgb_preds))

print(f'XGBoost  â€” MAE: {xgb_mae:.2f} | RMSE: {xgb_rmse:.2f}')

## 7. Compare & Evaluate

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'MAE':   [rf_mae, xgb_mae],
    'RMSE':  [rf_rmse, xgb_rmse]
})
print(results.to_string(index=False))

best_model_name = results.loc[results['MAE'].idxmin(), 'Model']
print(f'\nBest model by MAE: {best_model_name}')

In [ ]:
# Feature importance plot (XGBoost)
feat_imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('XGBoost Feature Importances')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted plot (sample: first 200 validation rows)
plt.figure(figsize=(14, 4))
plt.plot(y_val.values[:200], label='Actual', alpha=0.8)
plt.plot(xgb_preds[:200],    label='XGBoost Predicted', alpha=0.8)
plt.plot(rf_preds[:200],     label='RF Predicted', alpha=0.6)
plt.title('Actual vs Predicted (first 200 validation samples)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Save Model Artifacts

These files will be used by the Flask app.

In [ ]:
# Save to Google Drive so you can download them
OUTPUT_PATH = '/content/drive/MyDrive/FYP/models/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save both models
joblib.dump(xgb_model, OUTPUT_PATH + 'xgb_model.pkl')
joblib.dump(rf_model,  OUTPUT_PATH + 'rf_model.pkl')

# Save label encoders
joblib.dump(label_encoders, OUTPUT_PATH + 'label_encoders.pkl')

# Save feature column list
with open(OUTPUT_PATH + 'feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

# Save metrics
metrics = {
    'random_forest': {'MAE': round(rf_mae, 4),  'RMSE': round(rf_rmse, 4)},
    'xgboost':       {'MAE': round(xgb_mae, 4), 'RMSE': round(xgb_rmse, 4)}
}
with open(OUTPUT_PATH + 'model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Artifacts saved:')
for fname in os.listdir(OUTPUT_PATH):
    fsize = os.path.getsize(OUTPUT_PATH + fname) / 1024
    print(f'  {fname:35s} {fsize:.1f} KB')

## 10. LSTM Model (Deep Learning)

Train a 2-layer LSTM on 8-week sliding windows per (centre, meal) group.
The LSTM captures temporal demand patterns that tree-based models cannot.

**Input features per timestep (7 total):**
`log_orders`, `discount_pct`, `emailer_for_promotion`, `homepage_featured`,
`category_enc`, `cuisine_enc`, `center_type_enc`

**Architecture:** LSTM(7 → 128 hidden, 2 layers) → Linear(128→64) → ReLU → Linear(64→1)

In [ ]:
# PyTorch is pre-installed on Colab GPU runtimes — this is a no-op if already present
!pip install torch --quiet

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

# ── Hyper-parameters ─────────────────────────────────────────────────────────
SEQ_LEN    = 8
BATCH_SIZE = 2048
EPOCHS     = 25
LR         = 1e-3
HIDDEN     = 128
N_LAYERS   = 2

SEQ_FEATURES = [
    'log_orders', 'discount_pct',
    'emailer_for_promotion', 'homepage_featured',
    'category_enc', 'cuisine_enc', 'center_type_enc',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ── Model & Dataset definitions ───────────────────────────────────────────────
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=7, hidden_size=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

In [ ]:
# ── Create sliding-window sequences ─────────────────────────────────────────
# Reuses `df` and `label_encoders` from earlier cells
df_lstm = df.copy()
df_lstm['log_orders'] = np.log1p(df_lstm['num_orders'])

Xs, ys, ws = [], [], []
for _, grp in df_lstm.groupby(['center_id', 'meal_id']):
    grp  = grp.sort_values('week')
    vals = grp[SEQ_FEATURES].values.astype(np.float32)
    wks  = grp['week'].values
    for i in range(SEQ_LEN, len(vals)):
        Xs.append(vals[i - SEQ_LEN : i])
        ys.append(vals[i, 0])      # log_orders at target step
        ws.append(wks[i])

X      = np.array(Xs)
y      = np.array(ys, dtype=np.float32)
weeks  = np.array(ws)

# Same time-based split as XGBoost/RF (weeks 1–130 train, 131–145 val)
train_m = weeks <= TRAIN_END_WEEK
val_m   = weeks  > TRAIN_END_WEEK
X_tr, y_tr = X[train_m], y[train_m]
X_va, y_va = X[val_m],   y[val_m]
print(f'Sequences — Train: {len(X_tr):,}  |  Val: {len(X_va):,}')

In [ ]:
# ── Scale features ───────────────────────────────────────────────────────────
n_feat = len(SEQ_FEATURES)
scaler = MinMaxScaler()
scaler.fit(X_tr.reshape(-1, n_feat))       # fit on training data only

X_tr_s = scaler.transform(X_tr.reshape(-1, n_feat)).reshape(X_tr.shape)
X_va_s = scaler.transform(X_va.reshape(-1, n_feat)).reshape(X_va.shape)

# Scale the log_orders target using the same range as feature index 0
lo_min = float(scaler.data_min_[0])
lo_rng = float(scaler.data_max_[0] - scaler.data_min_[0])
y_tr_s = (y_tr - lo_min) / lo_rng
y_va_s = (y_va - lo_min) / lo_rng

tr_loader = DataLoader(SeqDataset(X_tr_s, y_tr_s), batch_size=BATCH_SIZE, shuffle=True)
va_loader = DataLoader(SeqDataset(X_va_s, y_va_s), batch_size=BATCH_SIZE * 2)
print('Scaling done.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
model     = LSTMForecaster(n_feat, HIDDEN, N_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.MSELoss()

best_val_loss = float('inf')
best_state    = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(X_tr)

    model.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb, yb in va_loader:
            xb, yb = xb.to(device), yb.to(device)
            va_loss += criterion(model(xb), yb).item() * len(yb)
    va_loss /= len(X_va)

    scheduler.step(va_loss)
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state    = {k: v.clone().cpu() for k, v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS}  train={tr_loss:.5f}  val={va_loss:.5f}')

model.load_state_dict(best_state)
model = model.cpu().eval()
print('Training complete.')

In [ ]:
# ── Evaluate — report MAE / RMSE in original order-count space ───────────────
preds_s = []
with torch.no_grad():
    for xb, _ in va_loader:
        preds_s.append(model(xb).numpy())
preds_s   = np.concatenate(preds_s)
preds_log = preds_s * lo_rng + lo_min
preds_ord = np.maximum(np.expm1(preds_log), 0)
true_ord  = np.maximum(np.expm1(y_va), 0)

lstm_mae  = mean_absolute_error(true_ord, preds_ord)
lstm_rmse = np.sqrt(mean_squared_error(true_ord, preds_ord))
print(f'LSTM  — MAE: {lstm_mae:.2f} | RMSE: {lstm_rmse:.2f}')

# Full model comparison
all_results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'LSTM'],
    'MAE':   [rf_mae,  xgb_mae,  lstm_mae],
    'RMSE':  [rf_rmse, xgb_rmse, lstm_rmse],
})
print(all_results.to_string(index=False))

In [ ]:
# ── Actual vs Predicted plot ──────────────────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(true_ord[:200],  label='Actual',         alpha=0.85, color='#00e5b0')
plt.plot(preds_ord[:200], label='LSTM Predicted',  alpha=0.85, color='#e064f7')
plt.title('LSTM — Actual vs Predicted (first 200 validation samples)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save LSTM artifacts to Google Drive ──────────────────────────────────────
# OUTPUT_PATH is already set in cell 8 (e.g. /content/drive/MyDrive/FYP/models/)

torch.save(model.state_dict(), OUTPUT_PATH + 'lstm_model.pt')

lstm_cfg_dict = {
    'input_size':   n_feat,
    'hidden_size':  HIDDEN,
    'num_layers':   N_LAYERS,
    'seq_len':      SEQ_LEN,
    'seq_features': SEQ_FEATURES,
    'lo_min':       lo_min,
    'lo_rng':       lo_rng,
}
with open(OUTPUT_PATH + 'lstm_config.json', 'w') as f:
    json.dump(lstm_cfg_dict, f, indent=2)

import joblib
joblib.dump(scaler, OUTPUT_PATH + 'lstm_scaler.pkl')

# Update model_metrics.json with LSTM results
with open(OUTPUT_PATH + 'model_metrics.json') as f:
    metrics = json.load(f)
metrics['lstm'] = {'MAE': round(lstm_mae, 4), 'RMSE': round(lstm_rmse, 4)}
with open(OUTPUT_PATH + 'model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:')
for fname in ['lstm_model.pt', 'lstm_config.json', 'lstm_scaler.pkl', 'model_metrics.json']:
    size = os.path.getsize(OUTPUT_PATH + fname) / 1024
    print(f'  {fname:35s} {size:.1f} KB')
print()
print('Download these 3 new files and place them in your local models/ folder:')
print('  lstm_model.pt, lstm_config.json, lstm_scaler.pkl')

## 9. Generate Predictions on test.csv (Optional)

This generates a `submission.csv` matching the Kaggle format.

In [ ]:
# Merge test set with metadata
test_df = test.merge(meal_info, on='meal_id', how='left')
test_df = test_df.merge(center_info, on='center_id', how='left')

# Apply same feature engineering (no lag features for test â€” use last known values from train)
test_df['discount_pct'] = ((test_df['base_price'] - test_df['checkout_price']) / test_df['base_price']).clip(lower=0)

for col in ['category', 'cuisine', 'center_type']:
    le = label_encoders[col]
    test_df[col + '_enc'] = le.transform(test_df[col].astype(str))

# For lag features on test set, use the last 3 weeks from train per (center_id, meal_id)
last_weeks = df.sort_values('week').groupby(['center_id', 'meal_id']).tail(3)
lag_lookup = df.sort_values('week').groupby(['center_id', 'meal_id'])['num_orders'].apply(list).reset_index()
lag_lookup.columns = ['center_id', 'meal_id', 'recent_orders']

test_df = test_df.merge(lag_lookup, on=['center_id', 'meal_id'], how='left')
test_df['lag_1']          = test_df['recent_orders'].apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 1 else np.nan)
test_df['lag_2']          = test_df['recent_orders'].apply(lambda x: x[-2] if isinstance(x, list) and len(x) >= 2 else np.nan)
test_df['lag_3']          = test_df['recent_orders'].apply(lambda x: x[-3] if isinstance(x, list) and len(x) >= 3 else np.nan)
test_df['rolling_mean_3'] = test_df['recent_orders'].apply(lambda x: np.mean(x[-3:]) if isinstance(x, list) and len(x) >= 1 else np.nan)
test_df['rolling_std_3']  = test_df['recent_orders'].apply(lambda x: np.std(x[-3:])  if isinstance(x, list) and len(x) >= 2 else 0)

test_df = test_df.fillna(0)

X_test      = test_df[FEATURE_COLS]
test_preds  = xgb_model.predict(X_test)
test_preds  = np.maximum(test_preds, 0)  # no negative orders

submission = pd.DataFrame({'id': test_df['id'], 'num_orders': test_preds.astype(int)})
submission.to_csv(OUTPUT_PATH + 'submission.csv', index=False)
print('submission.csv saved. Shape:', submission.shape)
display(submission.head())